# 03 — SfM-only hyperparameter tuning and ablations

This notebook consumes a completed frozen feature cache. It must not re-extract backbone features and must not use RevisitOP labels.

In [ ]:
%pip install -q -e .
from dataclasses import replace
from pathlib import Path

from cbir.cache import FeatureShardReader
from cbir.config import load_project_config
from cbir.data.sfm import Sfm30kMetadata
from cbir.evaluation import evaluate_sfm_verified_pairs, final_cls_descriptors_from_cache
from cbir.fusion import ReliabilityGatedFusion
from cbir.training import HeadTrainer

cfg = load_project_config(Path('configs/tuning.yaml'))
CACHE_DIR = Path('/content/cbir_cache/REPLACE_WITH_COMPLETED_CACHE_FOLDER')
reader = FeatureShardReader(CACHE_DIR)
metadata = Sfm30kMetadata.from_official_files(cfg.sfm.metadata_path, cfg.sfm.names_clusters_path)
val_ids = metadata.image_ids('val')
val_cases = metadata.build_validation_cases()

## Establish frozen CLS baseline

This is B0. Save its SfM R@1/R@5/R@10/MRR before trained heads are compared.

In [ ]:
baseline = final_cls_descriptors_from_cache(reader, val_ids)
print(evaluate_sfm_verified_pairs(baseline, val_ids, val_cases))

## Causal gate ablation

Run uniform, static, and reliability modes with the same layer set/dimension/training budget. Mean-vs-guided local pooling and last-only baselines are separate explicit configs, not hidden changes.

In [ ]:
def run_gate_mode(mode: str, output_root=Path('outputs/tuning')):
    fusion_cfg = replace(cfg.fusion, gate_mode=mode)
    head = ReliabilityGatedFusion.from_config(fusion_cfg)
    trainer = HeadTrainer(
        head=head,
        reader=reader,
        train_pairs=metadata.train_pairs,
        fusion_config=fusion_cfg,
        training_config=cfg.training,
        validation_cases=val_cases,
        validation_image_ids=val_ids,
        output_dir=output_root / mode,
    )
    return trainer.fit()

# histories = {mode: run_gate_mode(mode) for mode in ('uniform', 'static', 'reliability')}

Lock the selected configuration solely from these SfM results. Afterwards, run the selected static/RGMF variants with paired seeds and transfer the locked checkpoints to Notebook 04.